# Logistic Regression + ROC on Churn

**Week 3 · Lesson 5** &nbsp;·&nbsp; GBUS 564: Predictive Analytics &nbsp;·&nbsp; Fall 2026

<a target="_blank" href="https://colab.research.google.com/github/avorybryant/umw_gbus564_fall26/blob/main/notebooks/week3/lectures/05_logistic_regression_roc_churn.ipynb">
<img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
</a>

*Adapted from ISLP Ch. 4.7 Lab where relevant.*

## Setup

Same boilerplate as every notebook this term — see Lesson 8 if this is your
first time seeing it.

In [ ]:
# Environment setup — identical across every notebook in the course.
# Installs the course package (dataset loaders + helpers) from the public
# course repo; Colab already has numpy/pandas/scikit-learn, so it's quick.
%pip install -q matplotlib git+https://github.com/avorybryant/umw_gbus564_fall26.git

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn import set_config

set_config(display="diagram")
plt.rcParams["figure.figsize"] = (8, 5)

RANDOM_SEED = 564  # GBUS 564 — reuse this seed everywhere for reproducibility
rng = np.random.default_rng(RANDOM_SEED)

## 1. Business Problem

**Business question:** which customers are likely to churn, and — since a
retention team can't call everyone — which of those should actually get a
retention offer, given that a missed churner costs far more than an
unnecessary discount?

**Why it matters:** Week 1 Lesson 8 answered "who's likely to churn" with a
black-box random forest — accurate, but nothing a retention analyst could
take into a budget meeting. Lessons 3–4 built the alternative: a **logistic
regression** whose coefficients are a plain-English risk story, producing a
probability that a **threshold**, chosen by the relative cost of the two
mistakes, turns into a call list. This notebook fits that model for real.

## 2. Data

*Load, describe, and sanity-check the data before touching a model.*

Same dataset as Week 1 Lesson 8's pipeline walkthrough — **Telco Customer Churn**,
one row per customer, account/contract/service features plus a binary
`Churn` label. Committed as a static file (`data/telco_churn/`, see
`course_prep/course_data.md`); `load_churn()` in `gbus564_coursework.utils.course_data`
loads and cleans it the same way every lesson that touches this dataset.

In [ ]:
from gbus564_coursework.utils.course_data import load_churn

df = load_churn()
df.head()

In [ ]:
display(df.describe())

display(df.describe(exclude="number").T)

### Data Prep

Churn has real categorical features (`Contract`, `PaymentMethod`, ...) that
`LogisticRegression` can't consume directly, so the model below is a small
`Pipeline`: one-hot encode the categoricals, pass the numeric columns
(`tenure`, `MonthlyCharges`, `TotalCharges`) through unchanged, then fit.
Encoding inside the pipeline — rather than by hand — keeps `X_train` and
`X_test` transformed consistently and keeps the fitted encoder attached to
the model for Section 5's coefficient read-out.

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder

X, y = df.drop(columns="Churn"), df["Churn"]
categorical_cols = X.select_dtypes(exclude="number").columns

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_SEED, stratify=y
)

preprocess = ColumnTransformer(
    [("encode", OneHotEncoder(handle_unknown="ignore"), categorical_cols)],
    remainder="passthrough",
)

## 3. Model

*Baseline first — the reference every model must beat — then the model
under study.*

`DummyClassifier` (always predict the majority class, "stays") is the
baseline from Week 2. The model under study is **logistic regression** —
Lesson 3's log-odds machinery, fit here by maximum likelihood via
`scikit-learn` instead of by hand. `max_iter` is raised because the raw
numeric features sit on very different scales (`tenure` in months,
`TotalCharges` in thousands of dollars); that scale-sensitivity is exactly
what Lesson 8 covers in depth, so it's noted here, not fixed here.

In [ ]:
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline

baseline = DummyClassifier(strategy="most_frequent").fit(X_train, y_train)

model = Pipeline([
    ("prep", preprocess),
    ("clf", LogisticRegression(max_iter=5000, random_state=RANDOM_SEED)),
])
model.fit(X_train, y_train)

## 4. Evaluate

*Score against the baseline, not in isolation — a metric only means
something next to what it's beating.*

Four things, tying straight back to Lessons 3–4: the **probabilities**
p(X) the model actually outputs, a **confusion matrix at the default
t = 0.5 threshold**, the **ROC and precision–recall curves** traced out by
sweeping every possible threshold — the objects Week 2 introduced with the
stump, now built on a real, principled p(X) instead of a raw score — and
finally a **cost sweep** that uses dollars to pick one point on those
curves.

In [ ]:
proba = model.predict_proba(X_test)[:, 1]
proba[:10].round(3)

In [ ]:
from sklearn.metrics import ConfusionMatrixDisplay, accuracy_score, confusion_matrix

pred_default = (proba >= 0.5).astype(int)

baseline_acc = accuracy_score(y_test, baseline.predict(X_test))
model_acc = accuracy_score(y_test, pred_default)
print(f"Baseline accuracy: {baseline_acc:.2%}")
print(f"Model accuracy:    {model_acc:.2%}")

ConfusionMatrixDisplay(
    confusion_matrix(y_test, pred_default), display_labels=["Stayed", "Churned"]
).plot(cmap="Blues", colorbar=False)
plt.title("Confusion matrix at t = 0.5")
plt.tight_layout()

Nearly half of actual churners (170 of 374) land in the **false-negative**
cell — predicted "stayed," missed by the model. Lesson 4's argument was
that a missed churner usually costs more than an unnecessary retention
offer, so the next cells check what happens to that error if the threshold
moves.

In [ ]:
from sklearn.metrics import (
    PrecisionRecallDisplay, RocCurveDisplay, average_precision_score, roc_auc_score,
)

churn_rate = y_test.mean()
print(f"ROC AUC:           {roc_auc_score(y_test, proba):.3f}   (random guessing: 0.500)")
print(f"Average precision: {average_precision_score(y_test, proba):.3f}   "
      f"(random guessing: the churn rate, {churn_rate:.3f})")

fig, (ax_roc, ax_pr) = plt.subplots(1, 2, figsize=(12, 5))
RocCurveDisplay.from_predictions(y_test, proba, name="Logistic regression", ax=ax_roc)
ax_roc.plot([0, 1], [0, 1], linestyle="--", color="grey", label="Random guessing")
PrecisionRecallDisplay.from_predictions(y_test, proba, name="Logistic regression", ax=ax_pr)
ax_pr.axhline(churn_rate, linestyle="--", color="grey", label="Random guessing (churn rate)")

# Mark where the two thresholds compared in the next cell sit on each curve.
for t, color in [(0.5, "#1A73E8"), (0.3, "#D93025")]:
    tn, fp, fn, tp = confusion_matrix(y_test, (proba >= t).astype(int)).ravel()
    recall, fpr, precision = tp / (tp + fn), fp / (fp + tn), tp / (tp + fp)
    ax_roc.scatter(fpr, recall, s=80, color=color, zorder=3, label=f"t = {t}")
    ax_pr.scatter(recall, precision, s=80, color=color, zorder=3, label=f"t = {t}")

ax_roc.set_title("ROC — how well the model ranks churners above stayers")
ax_pr.set_title("Precision–recall — how many offers hit a real churner")
ax_roc.legend(loc="lower right")
ax_pr.legend(loc="upper right")
plt.tight_layout()

In [ ]:
from sklearn.metrics import confusion_matrix

# A lower threshold catches more true churners at the cost of more false
# positives — Lesson 4's cost argument, made concrete on real data.
for t in (0.5, 0.3):
    pred = (proba >= t).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_test, pred).ravel()
    recall = tp / (tp + fn)
    precision = tp / (tp + fp)
    print(f"t = {t:.1f}   recall (caught churners): {recall:.2%}   "
          f"precision (offer hit rate): {precision:.2%}   false positives: {fp}")

Dropping the threshold to 0.3 catches noticeably more true churners at the
cost of more wasted retention offers — precisely the tradeoff Lesson 4
described, and the two dots above show it as a move *along* each curve.
The two curves answer different questions. **ROC** (AUC ≈ 0.83) says how
well the model *ranks* churners above stayers, and doesn't depend on how
rare churn is. **Precision–recall** shows what the retention team actually
feels: precision is the share of offers that land on a real churner, and
its random-guessing floor is the 27% churn rate, not 0.5 — with churn the
minority class, that's the more honest picture of a call list.

What neither curve can do is say which point is *right*. Every point on
them is a tradeoff; picking one takes dollars.

**Choosing t by cost.** Week 2's assignment priced this exact decision: a
retention call costs about **\$40**, and a save is worth about **6 months
of a typical churner's monthly bill**. Calling a customer with churn
probability p pays off in expectation when p × save value > call cost, so
the break-even threshold is simply **call cost ÷ save value**. The cell
below computes that, then checks it empirically: sweep every threshold,
total the expected net benefit on the test set — saves × save value −
calls × call cost — and take the peak.

In [ ]:
call_cost = 40  # assumed, as in Week 2: ~15 min of agent time + overhead
save_value = 6 * X_train.loc[y_train == 1, "MonthlyCharges"].median()  # training rows only
breakeven = call_cost / save_value
print(f"call_cost=${call_cost}   save_value=${save_value:.0f}   "
      f"break-even threshold = {breakeven:.3f}")

thresholds = np.arange(0.01, 0.96, 0.01)
churned = (y_test == 1).to_numpy()

def net_benefit(t):
    call = proba >= t
    return (call & churned).sum() * save_value - call.sum() * call_cost

benefits = np.array([net_benefit(t) for t in thresholds])
best_t = thresholds[benefits.argmax()]

for label, t in [("default", 0.5), ("tried above", 0.3), ("cost-optimal", best_t)]:
    print(f"{label:>12}  t = {t:.2f}:  calls {int((proba >= t).sum()):>5,}   "
          f"net benefit ${net_benefit(t):>9,.0f}")

plt.plot(thresholds, benefits, color="grey", linewidth=2)
for t, color in [(0.5, "#1A73E8"), (0.3, "#D93025")]:
    plt.scatter(t, net_benefit(t), s=80, color=color, zorder=3, label=f"t = {t}")
plt.axvline(best_t, linestyle="--", color="black", linewidth=1,
            label=f"Cost-optimal t = {best_t:.2f}")
plt.xlabel("Threshold t")
plt.ylabel("Expected net benefit on test set ($)")
plt.title("Net benefit of the call list at every threshold")
plt.legend()
plt.tight_layout()

The break-even math says call anyone above roughly **8%** churn
probability; the empirical sweep peaks at **0.05** — the same neighborhood,
and the curve is nearly flat from about 0.03 to 0.15 (within ~\$3,000), so
the exact number matters far less than getting off 0.5. That move is worth
about **\$47,000** on these 1,409 test customers: the call list grows from
331 to 1,018, and catches 360 of the 374 actual churners instead of 204.
Because a wasted call (\$40) is so cheap next to a lost save (~\$478), the
right threshold sits far below even the 0.3 tried above.

One assumption is doing a lot of work: this treats every call to a churner
as a save. If only some fraction of called churners actually stay, the
effective save value shrinks by that fraction and the break-even threshold
rises with it — at a 30% save rate it's roughly 0.28, not 0.08. The method
doesn't change; the inputs do.

## 5. Interpret

*What is the model actually keying off of? Business-readable, not just
statistically significant.*

Logistic regression's payoff over Week 1's black box: every feature has
a **coefficient on the log-odds scale**. A positive coefficient pushes the
log-odds of churn up (more likely to churn); negative pushes it down. Sign
and rank are readable straight off the fit; converting a coefficient into a
plain "X% higher odds" statement is Lesson 10's job — here we just confirm
the model is keying off features a retention analyst would recognize.

In [ ]:
feature_names = model.named_steps["prep"].get_feature_names_out()
feature_names = [n.replace("encode__", "").replace("remainder__", "") for n in feature_names]
coefs = pd.Series(model.named_steps["clf"].coef_[0], index=feature_names)

top = pd.concat([coefs.sort_values().head(6), coefs.sort_values().tail(6)])
colors = ["#1A73E8" if v < 0 else "#D93025" for v in top]
top.plot.barh(color=colors, title="Coefficient on the log-odds of churn (red = pushes toward churn)")
plt.tight_layout()

`Contract_Month-to-month` and `InternetService_Fiber optic` push hardest
toward churn; two-year contracts and DSL push the other way — a story that
matches the retention team's intuition about no-commitment customers, built
directly from the fitted coefficients, not a post-hoc explanation bolted
onto a black box. `tenure`'s own coefficient is negative (more months,
lower churn odds) but small next to the 0/1 dummies above — it's measured
in whole months on its raw scale, not comparable size-for-size to a
one-hot flag. That comparability problem is precisely what Lesson 8's
"standardize first" fixes; it's a reason to be careful reading raw
coefficient *magnitude* side by side, not a reason to distrust the *sign*.

## 6. Recommend

**Recommendation:** score every active customer with this model, and route
customers above a **churn probability of about 0.05–0.10** — not 0.5 — to
the retention team. At \$40 a call against a ~\$478 save, that threshold
maximizes expected net benefit (about \$131,000 on the test set vs.
\$84,000 at 0.5), per Lesson 4's reasoning: a missed churner costs far more
than an unnecessary call, so the model should deliberately over-flag risk. `Contract` type is the clearest
lever the coefficients point to — early, month-to-month customers are the
segment to focus retention effort and messaging on.

**Caveats / next steps:** the raw numeric features are unscaled and no
regularization is applied yet, so a couple of coefficients above may be
less stable than they look — Lessons 6–9 (multicollinearity, then
ridge/lasso) revisit this exact fit once those tools exist. The threshold
is only as good as its inputs: it assumes every called churner is saved
and borrows Week 2's \$40 / 6-month figures, so a real deployment should
re-run the same sweep with the retention team's actual call cost, save
rate, and customer value.